# Lab 2.1 : XOR Classification with Sigmoid

## Objective
In this lab, you will build a small neural network **from scratch** to solve the **XOR** problem using:

- **sigmoid activation**
- **binary cross-entropy loss**
- **gradient descent**


## What you should understand at the end
- Why XOR cannot be solved well by a single linear model
- Why a hidden layer is needed
- How forward propagation works
- How binary cross-entropy is computed
- How backpropagation updates the parameters


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. XOR dataset
The XOR dataset contains 4 points:

- $(0,0) \rightarrow 0$
- $(0,1) \rightarrow 1$
- $(1,0) \rightarrow 1$
- $(1,1) \rightarrow 0$


In [ ]:
X = np.array([
    [0., 0.],
    [0., 1.],
    [1., 0.],
    [1., 1.]
])

y = np.array([
    [0.],
    [1.],
    [1.],
    [0.]
])

print("X shape:", X.shape)
print("y shape:", y.shape)
print(X)
print(y)


In [ ]:
# Visualize the XOR points
plt.figure(figsize=(5, 5))
for i in range(len(X)):
    if y[i, 0] == 0:
        plt.scatter(X[i, 0], X[i, 1], marker='o', s=120, label='Class 0' if i == 0 else "")
    else:
        plt.scatter(X[i, 0], X[i, 1], marker='s', s=120, label='Class 1' if i == 1 else "")

plt.xlim(-0.2, 1.2)
plt.ylim(-0.2, 1.2)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("XOR dataset")
plt.grid(True)
plt.legend()
plt.show()


## 2. Why XOR is difficult
A single linear classifier draws only one straight decision boundary.

XOR is **not linearly separable**, so we need a neural network with a hidden layer to learn a **nonlinear** boundary.


## 3. Network architecture

We will use:

- input size = **2**
- hidden size = **4**
- output size = **1**


## 4. Sigmoid activation

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Quick check: sigmoid(0) should be 0.5
print("sigmoid(0) =", sigmoid(0))  # expected: 0.5


## 5. Binary cross-entropy loss

### Important
- `y_true` contains labels 0 or 1
- `y_pred` contains predicted probabilities between 0 and 1
- use a small `eps` to avoid `log(0)`

In [ ]:
def binary_cross_entropy(y_true, y_pred, eps=1e-12):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(
        y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
    )


## 6. Derivative of sigmoid

If `a = sigmoid(z)`, then `sigmoid'(z) = a * (1 - a)`


In [ ]:
def sigmoid_derivative(a):
    # a is already sigmoid(z), so derivative = a*(1-a)
    return a * (1 - a)


## 7. Initialize the parameters

Use small random values for the weights and zeros for the biases.

Shapes:
- `W1`: `(2, 4)`
- `b1`: `(1, 4)`
- `W2`: `(4, 1)`
- `b2`: `(1, 1)`


In [ ]:
np.random.seed(42)

input_dim  = 2
hidden_dim = 4
output_dim = 1

W1 = 0.5 * np.random.randn(input_dim, hidden_dim)   # (2, 4)
b1 = np.zeros((1, hidden_dim))                       # (1, 4)
W2 = 0.5 * np.random.randn(hidden_dim, output_dim)  # (4, 1)
b2 = np.zeros((1, output_dim))                       # (1, 1)

print("W1 shape:", np.shape(W1))
print("b1 shape:", np.shape(b1))
print("W2 shape:", np.shape(W2))
print("b2 shape:", np.shape(b2))


## 8. Forward propagation

In [ ]:
def forward_propagation(X, W1, b1, W2, b2):
    Z1    = X @ W1 + b1       # (N, 4)
    A1    = sigmoid(Z1)       # (N, 4)
    Z2    = A1 @ W2 + b2      # (N, 1)
    y_hat = sigmoid(Z2)       # (N, 1)  -- output probabilities

    cache = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "y_hat": y_hat
    }
    return y_hat, cache


## 9. Backward propagation

Because we use **sigmoid + binary cross-entropy**, the output gradient simplifies to `y_hat - y`.

In [ ]:
def backward_propagation(X, y, W2, cache):
    N     = X.shape[0]
    A1    = cache["A1"]
    y_hat = cache["y_hat"]

    # --- output layer ---
    dZ2 = y_hat - y                                      # (N, 1)
    dW2 = (A1.T @ dZ2) / N                               # (4, 1)
    db2 = np.sum(dZ2, axis=0, keepdims=True) / N         # (1, 1)

    # --- hidden layer ---
    dA1 = dZ2 @ W2.T                                     # (N, 4)
    dZ1 = dA1 * sigmoid_derivative(A1)                   # (N, 4)
    dW1 = (X.T @ dZ1) / N                                # (2, 4)
    db1 = np.sum(dZ1, axis=0, keepdims=True) / N         # (1, 4)

    grads = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2
    }
    return grads


## 10. Parameter update

In [ ]:
def update_parameters(W1, b1, W2, b2, grads, lr):
    W1 = W1 - lr * grads["dW1"]
    b1 = b1 - lr * grads["db1"]
    W2 = W2 - lr * grads["dW2"]
    b2 = b2 - lr * grads["db2"]
    return W1, b1, W2, b2


## 11. Training loop


In [ ]:
lr     = 0.5
epochs = 10000

loss_history = []

for epoch in range(epochs):
    # forward pass
    y_hat, cache = forward_propagation(X, W1, b1, W2, b2)

    # compute loss
    loss = binary_cross_entropy(y, y_hat)

    # backward pass
    grads = backward_propagation(X, y, W2, cache)

    # update parameters
    W1, b1, W2, b2 = update_parameters(W1, b1, W2, b2, grads, lr)

    loss_history.append(loss)

    if epoch % 1000 == 0:
        print(f"Epoch {epoch:5d} | Loss = {loss:.6f}")


## 12. Plot the loss


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("Training loss")
plt.grid(True)
plt.show()


## 13. Final prediction

Convert probabilities to class labels using threshold 0.5.

In [ ]:
y_hat, _ = forward_propagation(X, W1, b1, W2, b2)

y_pred = (y_hat >= 0.5).astype(int)

print("Predicted probabilities:")
print(np.round(y_hat, 4))

print("\nPredicted labels:")
print(y_pred.ravel())

print("\nTrue labels:")
print(y.ravel())

accuracy = np.mean(y_pred == y)
print(f"\nAccuracy: {accuracy * 100:.2f}%")


## 14. Decision boundary visualization


In [ ]:
xx, yy = np.meshgrid(
    np.linspace(-0.2, 1.2, 200),
    np.linspace(-0.2, 1.2, 200)
)
grid = np.c_[xx.ravel(), yy.ravel()]

grid_pred, _ = forward_propagation(grid, W1, b1, W2, b2)
grid_pred = grid_pred.reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, grid_pred, levels=20, alpha=0.6)
plt.colorbar(label="Predicted probability")

for i in range(len(X)):
    if y[i, 0] == 0:
        plt.scatter(X[i, 0], X[i, 1], marker='o', s=120, edgecolors='k', label='Class 0' if i == 0 else "")
    else:
        plt.scatter(X[i, 0], X[i, 1], marker='s', s=120, edgecolors='k', label='Class 1' if i == 1 else "")

plt.xlim(-0.2, 1.2)
plt.ylim(-0.2, 1.2)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Decision boundary for XOR")
plt.legend()
plt.show()


## 15. Questions

1. **Why can a single linear classifier not solve XOR?**

   XOR est non-linéairement séparable : il n'existe aucune droite qui puisse séparer les points de classe 0 `{(0,0),(1,1)}` des points de classe 1 `{(0,1),(1,0)}`. Un classifieur linéaire ne peut tracer qu'une seule frontière de décision linéaire, ce qui est insuffisant pour ce problème.

2. **Why do we use sigmoid at the output?**

   Sigmoid comprime la sortie dans l'intervalle `]0,1[`, ce qui l'interprète comme une probabilité. Couplée à la binary cross-entropy, elle permet de mesurer la divergence entre les probabilités prédites et les vraies étiquettes binaires, et son gradient avec la BCE se simplifie à `y_hat - y`.

3. **What happens if you change the learning rate?**

   - **LR trop grand (ex : 5.0)** : les mises à jour oscillent, la loss peut diverger.
   - **LR trop petit (ex : 0.01)** : la convergence est très lente et nécessite beaucoup plus d'époques.
   - **LR optimal (~0.5)** : convergence rapide et stable vers une loss proche de zéro et une accuracy de 100%.
